# Czyszczenie danych: z surowych CSV do tabel gotowych do analizy

Surowe dane mają trzy typowe problemy:
- daty zapisane jako tekst,
- literówki w nazwach kolumn i wartości,
- geolokalizację w postaci 1 mln wierszy zamiast jednego punktu na kod pocztowy.

Ten notebook naprawia je i nadpisuje odpowiednie tabele w bazie.

**Wejście:** `data/raw/` — orders, products, payments, sellers, geolocation

**Wyjście:** `data/processed/*.csv` + nadpisane tabele w `data/olist.db`


> Kolejność ma znaczenie: `01_import_data.ipynb` buduje bazę z danych surowych,
> ten notebook podmienia wybrane tabele na wersje oczyszczone.

In [ ]:
import pandas as pd
import os

In [2]:
path = '../data/raw/'

orders = pd.read_csv(path + 'olist_orders_dataset.csv')
products = pd.read_csv(path + 'olist_products_dataset.csv')
payments = pd.read_csv(path + 'olist_order_payments_dataset.csv')
sellers = pd.read_csv(path + 'olist_sellers_dataset.csv')

## Daty i czas dostawy

Wszystkie kolumny z datami wczytują się jako tekst, więc nie da się na nich liczyć różnic. Po konwersji na typ datetime powstaje `delivery_days` — kluczowa metryka całego projektu, bo to na niej opiera się analiza zależności między czasem dostawy a oceną klienta. Wartości powyżej 100 dni sprawdzam osobno: to realne przypadki (63 zamówienia), nie błędy danych, więc zostają w zbiorze.

In [3]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
  ]

orders[date_cols] = orders[date_cols].apply(pd.to_datetime)

orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [4]:
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
    ).dt.days

orders['delivery_days'].describe()

count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64

In [5]:
orders[orders['delivery_days'] > 100][['order_id', 'delivery_days', 'order_status']]

,order_id,delivery_days,order_status
1621,a4efaffc506a395c9cea7402b078c1e5,110.0,delivered
3077,8b7fd198ad184563c231653673e75a7f,105.0,delivered
3202,4f39a94d6e474819d898d6df7d394996,143.0,delivered
4666,b31c7dea63bb08f8cdd1ec32514ccf0b,132.0,delivered
10383,3602a80b09d914236f74c733631f3b8b,106.0,delivered
...,...,...,...
86520,ed8e9faf1b75f43ee027103957135663,173.0,delivered
89130,285ab9426d6982034523a855f55a885e,194.0,delivered
92212,29c3b79aace1b72a82b1232bf494e16f,133.0,delivered
95136,17cc6728043d53cc948551dfbf0a338b,142.0,delivered


## Czyszczenie wartości i nazw kolumn

Trzy niezależne poprawki: `not_defined` w metodzie płatności zamieniam na `unknown` (czytelniejsze w raportach), w tabeli produktów prostuję literówkę z pliku źródłowego (`lenght` na `length`) i uzupełniam brakujące kategorie, a w tabeli sprzedawców ujednolicam zapis miast.

In [6]:
payments['payment_type'] = payments['payment_type'].replace('not_defined', 'unknown')

payments['payment_type'].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
unknown            3
Name: count, dtype: int64

In [ ]:
products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length',
    'product_lenght_cm': 'product_length_cm'
  }, inplace=True)

products['product_category_name'] = products['product_category_name'].fillna('unknown')

products.isnull().sum()

product_id                      0
product_category_name           0
product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [ ]:
sellers['seller_city'].value_counts().head(20)

seller_city
sao paulo                694
curitiba                 127
rio de janeiro            96
belo horizonte            68
ribeirao preto            52
guarulhos                 50
ibitinga                  49
santo andre               45
campinas                  41
maringa                   40
sao jose do rio preto     33
sao bernardo do campo     32
osasco                    32
sorocaba                  32
brasilia                  28
porto alegre              28
londrina                  26
goiania                   23
joinville                 22
blumenau                  21
Name: count, dtype: int64

In [9]:
# Świadome ograniczenie: poprawiam tylko najczęstsze literówki (top wolumen).
# Kolumna seller_city nie wchodzi do żadnej analizy (agregacje idą po seller_state),
# więc pełne mapowanie po zip_code_prefix nie jest tu warte kosztu.
sellers['seller_city'] = sellers['seller_city'].str.strip().str.lower()

sellers['seller_city'] = sellers['seller_city'].replace({
    'rio de janeiro, rio de janeiro, brasil': 'rio de janeiro',
    'riberao preto': 'ribeirao preto',
    '04482255': 'sao paulo',
    'sao pauo': 'sao paulo',
    'pao paulo': 'sao paulo',
    'sp': 'sao paulo',
    'sao paulo/ sao paulo': 'sao paulo'
  })

sellers['seller_city'].value_counts().tail(100)

seller_city
carapicuiba / sao paulo    1
centro                     1
parana                     1
bombinhas                  1
orlandia                   1
                          ..
aparecida de goiania       1
bandeirantes               1
vitoria de santo antao     1
palotina                   1
leme                       1
Name: count, Length: 100, dtype: int64

In [10]:
sellers.isnull().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

## Zapis oczyszczonych danych

Oczyszczone tabele trafiają do `data/processed/` (wersja do wglądu i do Power Query) oraz nadpisują tabele w `data/olist.db`, na której pracują dalsze notebooki i zapytania SQL. Osobno agreguję geolokalizację: 1 mln wierszy z powtarzającymi się kodami pocztowymi redukuję do jednego uśrednionego punktu na kod (~19 tys. wierszy).

In [11]:
os.makedirs('../data/processed', exist_ok=True)

orders.to_csv('../data/processed/orders_clean.csv', index=False)
payments.to_csv('../data/processed/payments_clean.csv', index=False)
products.to_csv('../data/processed/products_clean.csv', index=False)
sellers.to_csv('../data/processed/sellers_clean.csv', index=False)

In [12]:
# Geolocation: 1M wierszy, każdy zip code występuje wielokrotnie z drobnymi różnicami GPS.
# Redukcja do jednego reprezentatywnego punktu per zip code (średnia lat/lng).
geo = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')

geo_clean = geo.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean().reset_index()
geo_clean.to_csv('../data/processed/geolocation_clean.csv', index=False)

print(f'Geolocation: {len(geo)} wierszy → {len(geo_clean)} unikalnych zip codes')

Geolocation: 1000163 wierszy → 19015 unikalnych zip codes


In [ ]:
import sqlite3

conn = sqlite3.connect('../data/olist.db')

orders.to_sql('orders', conn, if_exists='replace', index=False)
payments.to_sql('payments', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
sellers.to_sql('sellers', conn, if_exists='replace', index=False)
geo_clean.to_sql('geolocation', conn, if_exists='replace', index=False)

conn.close()
print('Import ukończony')

Import ukończony
